In [1]:
from pypdf import PdfReader
reader = PdfReader("data/icd10cm_guidelines_2026.pdf")
print(len(reader.pages))

121


checking a page

In [2]:
page_15 = reader.pages[14]
page_15_text = page_15.extract_text()
print(page_15_text[:800])

ICD-10-CM Official Guidelines for Coding and Reporting 
FY 2026 
Page 15 of 121 
 
11. Impending or Threatened Condition 
Code any condition described at the time of discharge as “impending” or “threatened” 
as follows: 
If it did occur, code as confirmed diagnosis. 
If it did not occur, reference the Alphabetic Index to determine if the condition has a 
subentry term for “impending” or “threatened” and also reference main term entries 
for “Impending” and for “Threatened.” 
If the subterms are listed, assign the given code. 
If the subterms are not listed, code the existing underlying condition(s) and not the 
condition described as impending or threatened. 
12. Reporting Same Diagnosis Code More than Once 
Each unique ICD-10-CM diagnosis code may be reported only once for an encounter. 



In [3]:
print(reader.pages[106].extract_text()[:300])

ICD-10-CM Official Guidelines for Coding and Reporting 
FY 2026 
Page 107 of 121 
 
indicating the patient is experiencing homelessness would support 
assignment of a code from subcategory Z59.0-, Homelessness. 
 
For social determinants of health classified to chapter 21, such as 
information found


extracting each page data into a list

In [4]:
pages_data = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    pages_data.append({"page": page_number, "text": text})

print(len(pages_data))
print(pages_data[14]["page"])
print(pages_data[14]["text"][:300])

121
15
ICD-10-CM Official Guidelines for Coding and Reporting 
FY 2026 
Page 15 of 121 
 
11. Impending or Threatened Condition 
Code any condition described at the time of discharge as “impending” or “threatened” 
as follows: 
If it did occur, code as confirmed diagnosis. 
If it did not occur, reference t


removing unecessary text or spaces and creating new list

In [5]:
def clean_page_text(text):
    clean_lines = []
    for line in text.split("\n"):
        line = line.strip()
        if line == "":
            continue
        if line == "ICD-10-CM Official Guidelines for Coding and Reporting":
            continue
        if line == "FY 2026":
            continue
        if line.startswith("Page ") and "of 121" in line:
            continue
        clean_lines.append(line)
    return "\n".join(clean_lines)

print(clean_page_text(pages_data[14]["text"])[:300])

11. Impending or Threatened Condition
Code any condition described at the time of discharge as “impending” or “threatened”
as follows:
If it did occur, code as confirmed diagnosis.
If it did not occur, reference the Alphabetic Index to determine if the condition has a
subentry term for “impending” o


cleaning all the pages data and also extracting headers and charachters left

In [6]:
for item in pages_data:
    item["text"] = clean_page_text(item["text"])

leftover_headers = 0
total_chars = 0

for item in pages_data:
    if "of 121" in item["text"]:
        leftover_headers = leftover_headers + 1
    total_chars = total_chars + len(item["text"])

print(leftover_headers)
print(total_chars)

0
301713


chunking page by page

In [7]:
def make_chunks(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = start + chunk_size - overlap
    return chunks

test = make_chunks("ABCDEFGHIJKLMNOPQRST", chunk_size=10, overlap=3)
print(test)

['ABCDEFGHIJ', 'HIJKLMNOPQ', 'OPQRST']


making 4 label for every page

In [8]:
all_chunks = []

for item in pages_data:
    page_chunks = make_chunks(item["text"])
    for chunk_text in page_chunks:
        all_chunks.append({
            "chunk_id": len(all_chunks),
            "source": "ICD-10-CM Official Guidelines FY2026",
            "page": item["page"],
            "text": chunk_text,
        })

print(len(all_chunks))
print(all_chunks[40])

404
{'chunk_id': 40, 'source': 'ICD-10-CM Official Guidelines FY2026', 'page': 6, 'text': '.................................................. 112\n1. Outpatient Surgery ................................................................................................................ 112\n2. Observation Stay ................................................................................................................... 112\nB. Codes from A00.0 through T88.9, Z00-Z99, U00-U85 .............................................................. 113\nC. Accurate reporting of ICD-10-CM diagnosis codes ................................................................... 113\nD. Codes that describe symptoms and signs .................................................................................... 113\nE. Encounters for circumstances other than a disease or injury ...................................................... 113\nF. Level of Detail in Coding ............................................

removing content page- dotted lines

In [9]:
all_chunks = []

for item in pages_data:
    page_chunks = make_chunks(item["text"])
    for chunk_text in page_chunks:
        if "........" in chunk_text:
            continue
        all_chunks.append({
            "chunk_id": len(all_chunks),
            "source": "ICD-10-CM Official Guidelines FY2026",
            "page": item["page"],
            "text": chunk_text,
        })

print(len(all_chunks))
print(all_chunks[0]["page"])
print(all_chunks[0]["text"][:300])

365
1
FY 2026 -- UPDATED April 1, 2026
(April 1, 2026 - September 30, 2026)
Narrative changes appear in bold text
Items underlined have been moved within the guidelines since the October 2025, FY 2026 version
Italics are used to indicate revisions to heading changes
The Centers for Medicare and Medicaid S


saving all chunks

In [10]:
import json

with open("data/icd_guideline_chunks.json", "w") as f:
    json.dump(all_chunks, f)

with open("data/icd_guideline_chunks.json", "r") as f:
    loaded_chunks = json.load(f)

print(len(loaded_chunks))
print(loaded_chunks[0]["page"], loaded_chunks[0]["source"])

365
1 ICD-10-CM Official Guidelines FY2026


# sentence vectorization using bge small model
check on a sentence

In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

test_sentences = [
    "When laterality is not documented, assign the unspecified side code",
    "The doctor did not write which side of the body is affected",
    "Diabetes mellitus in pregnancy",
]

test_vectors = embed_model.encode(test_sentences)

print(test_vectors.shape)
print(cosine_similarity(test_vectors).round(2))

/opt/miniconda3/envs/dl_projects/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17382.73it/s]


(3, 384)
[[1.   0.63 0.48]
 [0.63 1.   0.53]
 [0.48 0.53 1.  ]]


In [12]:
import numpy as np

card_texts = [chunk["text"] for chunk in loaded_chunks]

chunk_vectors = embed_model.encode(card_texts, show_progress_bar=True)
print(chunk_vectors.shape)

np.save("data/icd_guideline_vectors.npy", chunk_vectors)

reloaded_vectors = np.load("data/icd_guideline_vectors.npy")
print(reloaded_vectors.shape)

check = embed_model.encode([loaded_chunks[15]["text"]])
print(cosine_similarity(check, reloaded_vectors[15:16]).round(4))

Batches: 100%|██████████| 12/12 [00:02<00:00,  5.88it/s]


(365, 384)
(365, 384)
[[1.]]


finding similar 3 highest score

In [13]:
query = "When laterality is not documented, which code should be assigned?"

query_vector = embed_model.encode([query])

scores = cosine_similarity(query_vector, reloaded_vectors)[0]
print(scores.shape)

top_ids = np.argsort(-scores)[:3]
print(top_ids)

for i in top_ids:
    print(round(scores[i], 3), "| page", loaded_chunks[i]["page"])
    print(loaded_chunks[i]["text"][:200])
    print("---")

(365,)
[ 33 137  34]
0.798 | page 15
 two different conditions classified to the same ICD-10-CM diagnosis code.
13. Laterality
Some ICD-10-CM codes indicate laterality, specifying whether the condition occurs on
the left, right or is bil
---
0.784 | page 49
 classification does not
distinguish laterality (i.e., subcategories H40.10 and H40.20), assign a
code for the type of glaucoma for each eye with the seventh character for
the specific glaucoma stage 
---
0.764 | page 15
unilateral code for the side where the condition still exists
(e.g., cataract surgery performed on each eye in separate encounters). The bilateral code
would not be assigned for the subsequent encount
---


Making function for the same - finiding similarity top 3 scores

In [14]:
def search_coding_guidelines(query, k=3):
    query_vector = embed_model.encode([query])
    scores = cosine_similarity(query_vector, reloaded_vectors)[0]
    top_ids = np.argsort(-scores)[:k]

    results = []
    for i in top_ids:
        results.append({
            "source": loaded_chunks[i]["source"],
            "page": loaded_chunks[i]["page"],
            "score": round(float(scores[i]), 3),
            "text": loaded_chunks[i]["text"],
        })
    return results

answer = search_coding_guidelines("When laterality is not documented, which code should be assigned?")
for r in answer:
    print(r["score"], "| page", r["page"])

0.798 | page 15
0.784 | page 49
0.764 | page 15


In [15]:
my_answer = search_coding_guidelines("How to code a condition that is both acute and chronic?")
for r in my_answer:
    print(r["score"], "| page", r["page"])
    print(r["text"][:200])
    print("---")

0.817 | page 119
s, and this
diagnosis is based on symptoms or clinical findings that were present on
admission, assign “Y”.
If the final diagnosis contains an impending or threatened diagnosis, and this
diagnosis is 
---
0.792 | page 119
If a single code only identifies the chronic condition and not the acute
exacerbation (e.g., acute exacerbation of chronic leukemia), assign “Y.”
Conditions documented as possible, probable, suspected
---
0.781 | page 14
Multiple codes may be needed for sequela, complication codes and obstetric codes to
more fully describe a condition. See the specific guidelines for these conditions for
further instruction.
8. Acute 
---


In [16]:
prefix = "Represent this sentence for searching relevant passages: "
test = search_coding_guidelines(prefix + "How to code a condition that is both acute and chronic?")
for r in test:
    print(r["score"], "| page", r["page"])

0.798 | page 14
0.794 | page 119
0.763 | page 119


In [18]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def search_coding_guidelines(query, k=3):
    query_vector = embed_model.encode([QUERY_PREFIX + query])
    scores = cosine_similarity(query_vector, reloaded_vectors)[0]
    top_ids = np.argsort(-scores)[:k]

    results = []
    for i in top_ids:
        results.append({
            "source": loaded_chunks[i]["source"],
            "page": loaded_chunks[i]["page"],
            "score": round(float(scores[i]), 3),
            "text": loaded_chunks[i]["text"],
        })
    return results

for q in ["When laterality is not documented, which code should be assigned?",
          "How to code a condition that is both acute and chronic?"]:
    print("Q:", q)
    for r in search_coding_guidelines(q):
        print("  ", r["score"], "| page", r["page"])

Q: When laterality is not documented, which code should be assigned?
   0.774 | page 15
   0.771 | page 49
   0.75 | page 15
Q: How to code a condition that is both acute and chronic?
   0.798 | page 14
   0.794 | page 119
   0.763 | page 119
